# Predicting Customer Dissatisfaction Before It's Posted

*Binary review-score classification on 99k Brazilian e-commerce orders (Olist), using only pre-delivery signals.*

**Notebook 1 of a planned series - this one covers the EDA.** The goal is to understand the data well enough to know which features are worth building and where to draw the line between a "good" and "bad" review. Modeling comes in a later notebook.

I work through the data in five passes: check the **shape**, check the **quality**, look at each feature **on its own**, look at each feature **against the target**, then look at the features **together**.

---

## Problem statement

Predict whether a customer leaves a good or bad review on an Olist order. Binary classification.

- **Grain: One row = one order.**
- **Target:** `review_score` (1–5 stars), collapsed into good/bad. I haven't picked the cut-off yet. I'll decide it later in this notebook, once I've seen how the scores are spread out and how they move with the features.
- **What I'm allowed to use:** only things known at or before delivery - delivery lateness, price, freight, product category. Anything that only exists after the customer writes the review is off limits, since the model wouldn't have it at prediction time. That's classical data leakage.

If a step doesn't help predict `review_score` from pre-review data, it's out of scope here.

## Setup

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = "../data/raw/"


## Understanding the data

Load each table. For each one I check the row and column counts, the data types, and how it joins to the others that is which keys link orders to reviews, items, products, and customers.

---

### Load tables

In [ ]:
df_customers = pd.read_csv(DATA_DIR+"olist_customers_dataset.csv")
df_geolocation = pd.read_csv(DATA_DIR + "olist_geolocation_dataset.csv")
df_order_items = pd.read_csv(DATA_DIR + "olist_order_items_dataset.csv")
df_order_payments = pd.read_csv(DATA_DIR+"olist_order_payments_dataset.csv")
df_order_reviews = pd.read_csv(DATA_DIR + "olist_order_reviews_dataset.csv")
df_orders = pd.read_csv(DATA_DIR + "olist_orders_dataset.csv")
df_products = pd.read_csv(DATA_DIR + "olist_products_dataset.csv")
df_sellers = pd.read_csv(DATA_DIR + "olist_sellers_dataset.csv")

### Shape, dtypes, nulls

In [ ]:
print('shape of customers dataset:', df_customers.shape)
print('shape of geolocation dataset:', df_geolocation.shape)
print('shape of order items dataset:', df_order_items.shape)
print('shape of order payments dataset:', df_order_payments.shape)
print('shape of order reviews dataset:', df_order_reviews.shape)
print('shape of orders dataset:', df_orders.shape)
print('shape of products dataset:', df_products.shape)
print('shape of sellers dataset:', df_sellers.shape)

In [ ]:
print('info of customers dataset:', df_customers.info(), '\n')
print('info of geolocation dataset:', df_geolocation.info(), '\n')
print('info of order items dataset:', df_order_items.info(), '\n')
print('info of order payments dataset:', df_order_payments.info(), '\n')
print('info of order reviews dataset:', df_order_reviews.info(), '\n')
print('info of orders dataset:', df_orders.info(), '\n')
print('info of products dataset:', df_products.info(), '\n')
print('info of sellers dataset:', df_sellers.info(), '\n')

| Dataset | Shape (Rows × Cols) | Data Types Breakdown | Total Null Cells (%) | Tables with Nulls |
| :--- | :---: | :--- | :---: | :---: |
| **customers** | (99,441, 5) | `object/str`: 4, `int64`: 1 | 0 (0.0%) | None |
| **geolocation** | (1,000,163, 5) | `object/str`: 2, `float64`: 2, `int64`: 1 | 0 (0.0%) | None |
| **order_items** | (112,650, 7) | `object/str`: 4, `float64`: 2, `int64`: 1 | 0 (0.0%) | None |
| **order_payments** | (103,886, 5) | `object/str`: 2, `int64`: 2, `float64`: 1 | 0 (0.0%) | None |
| **order_reviews** | (99,224, 7) | `object/str`: 6, `int64`: 1 | 145,903 (21.0%) | `review_comment_title`, `review_comment_message` |
| **orders** | (99,441, 8) | `object/str`: 8 | 4,908 (0.6%) | `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date` |
| **products** | (32,951, 9) | `float64`: 7, `object/str`: 2 | 2,448 (0.8%) | `product_category_name`, dimensions, photo counts |
| **sellers** | (3,095, 4) | `object/str`: 3, `int64`: 1 | 0 (0.0%) | None |

### Preview Rows

---

In [ ]:
display('Orders:', df_orders.head(3))
display('Order Items:', df_order_items.head(3))
display('Order Payments:', df_order_payments.head(3))
display('Order Reviews:', df_order_reviews.head(3))

display('Customers:', df_customers.head(3))
display('Sellers:', df_sellers.head(3))
display('Products:', df_products.head(3))

display('Geolocation:', df_geolocation.head(3))

### How the tables connect (a guess, not yet checked)

This is my working guess at how the tables join. I built it from two clues: columns that share a name across tables, and what the row-count gaps hint at. I haven't verified it yet that's the job of the quality checks next.

- **orders - customers:** `customer_id` (same column in both).
- **orders - order_items:** `order_id`. Probably one-to-many - order_items has more rows (112,650) than orders (99,441), so `order_id` must repeat. Fits the idea of one order holding several items.
- **orders - order_payments:** `order_id`. Probably one-to-many too - order_payments (103,886) has more rows than orders. Makes sense: one order can be paid in more than one way or in installments.
- **orders - order_reviews:** `order_id`. Row counts are close (99,224 vs 99,441) but not equal either way, so I can't guess the relationship from counts alone. I check this directly below.
- **order_items - products:** `product_id` (same column).
- **order_items - sellers:** `seller_id` (same column).

**Skipping `df_geolocation`.** It's a zip-code to latitide/lng lookup, not tied directly to orders or reviews, and the location info I actually need (city/state) is already in the customers and sellers tables. Leaving it out keeps the merge simpler.

---

**Why check order_reviews right now, instead of waiting for the Quality phase?**

`review_score` is the prediction target. If its join to orders is broken in some way - missing for some orders, or duplicated for others - and I merge without knowing that, it could either silently drop my ability to train (missing labels) or silently duplicate rows (double-counting an order). That is a bigger risk than getting a feature column slightly wrong, so this one relationship gets verified immediately, ahead of the rest of the Quality checks on the other tables.

---

In [ ]:
df_order_reviews.isnull().sum()

`review_comment_title` (88.3% missing) and `review_comment_message` (58.7% missing) have high missing rates, whereas `review_score` has zero missing values. target label integrity is completely intact

### Checking the target join: orders and order_reviews


In [ ]:
print('number of unique order_id in order reviews dataset:', df_order_reviews['order_id'].nunique())
print('number of unique order_id in orders dataset:', df_orders['order_id'].nunique())

In [ ]:
len(df_order_reviews)

In [ ]:
df_order_reviews["order_id"].duplicated().sum()

99,441 − 98,673 = **768 orders have no review at all**  no `review_score` label. These get dropped once merged, since I can't train without a target.

**551 `order_id`s appear on more than one review row.** Before merging, I dedupe `order_reviews` down to one row per order (keeping the most recent by timestamp), otherwise these orders would show up twice in the merged table.


In [ ]:
df_order_reviews = df_order_reviews.astype({'review_answer_timestamp': 'datetime64[ns]'}).sort_values(
    by="review_answer_timestamp",
).drop_duplicates(subset=["order_id"], keep="last")
print("number of rows in order reviews dataset after removing duplicates:", len(df_order_reviews),)


98,673 rows now, which matches the unique-count from earlier. The dedupe worked and we got one review per order, keeping the most recent `review_answer_timestamp`.

## Data Quality Checks

Duplicates, whether keys are unique, and whether foreign keys actually match up across tables.

In [ ]:
print("number of duplicate rows in customers dataset:", df_customers.duplicated().sum())
print("number of duplicate rows in order items dataset:", df_order_items.duplicated().sum())
print("number of duplicate rows in order payments dataset:", df_order_payments.duplicated().sum())
print("number of duplicate rows in order reviews dataset:", df_order_reviews.duplicated().sum())
print("number of duplicate rows in orders dataset:", df_orders.duplicated().sum())
print("number of duplicate rows in products dataset:", df_products.duplicated().sum())
print("number of duplicate rows in sellers dataset:", df_sellers.duplicated().sum())

**Full-row duplicate check.** This looks for rows that are identical across *every* column, not just a repeated key. It catches accidental double-loading that a key check alone would miss.

**Result: 0 duplicate rows in every table.** Nothing was loaded twice - clean on this front.

---

In [ ]:
print('Ratio of unique values, customer_id in customers:', df_customers['customer_id'].nunique()/len(df_customers))
print('Ratio of unique values, order_id in order_items:', df_order_items['order_id'].nunique()/len(df_order_items))
print('Ratio of unique values, order_id in order_payments:', df_order_payments['order_id'].nunique()/len(df_order_payments))
print('Ratio of unique values, seller_id in sellers:', df_sellers['seller_id'].nunique()/len(df_sellers))
print('Ratio of unique values, product_id in products:', df_products['product_id'].nunique()/len(df_products))
print('Ratio of unique values, order_id in order_reviews:', df_order_reviews['order_id'].nunique()/len(df_order_reviews))
print('Ratio of unique values, order_id in orders:', df_orders['order_id'].nunique()/len(df_orders))

**Uniqueness ratio (unique values ÷ total rows; 1.0 means fully unique):**

- customer_id in customers: 1.0
- order_id in order_items: 0.876
- order_id in order_payments: 0.957
- seller_id in sellers: 1.0
- product_id in products: 1.0
- order_id in order_reviews: 1.0 - confirms the dedupe worked
- order_id in orders: 1.0

This confirms the one-to-many guess for order_items and order_payments (`order_id` repeats in both). customers, sellers, products, reviews, and orders each have a valid single-column primary key.

**Still to check:** this only tells me each key is unique *within* its own table. It doesn't tell me whether every foreign key value actually exists in the parent table — e.g. does every `product_id` in order_items exist in products? That's referential integrity, and I check it next with `.isin()`.

---

In [ ]:
print('Referential integrity for customer_id in orders:', df_orders["customer_id"].isin(df_customers["customer_id"]).all())
print('Referential integrity for order_id in order_items:', df_order_items["order_id"].isin(df_orders["order_id"]).all())
print('Referential integrity for order_id in order_payments:', df_order_payments["order_id"].isin(df_orders["order_id"]).all())
print('Referential integrity for order_id in order_reviews:', df_order_reviews["order_id"].isin(df_orders["order_id"]).all())
print('Referential integrity for product_id in order_items:', df_order_items["product_id"].isin(df_products["product_id"]).all())
print('Referential integrity for seller_id in order_items:', df_order_items["seller_id"].isin(df_sellers["seller_id"]).all())

**Referential integrity: all True.**

Every foreign key I checked points to a real row in its parent table, with no orphans:

- orders.customer_id - customers.customer_id
- order_items.order_id - orders.order_id
- order_payments.order_id - orders.order_id
- order_reviews.order_id - orders.order_id
- order_items.product_id - products.product_id
- order_items.seller_id - sellers.seller_id

Together with the uniqueness ratios above, the join map is now verified rather than guessed. Safe to merge.

---

### Table relationships (verified)

![Table relationships](assets/table_relationships.png)

Green edges (1:1 or many:1) can be merged onto `orders` directly. Orange edges (1:many for order_items, order_payments) can't be merged directly without breaking the "one row per order" rule, so I aggregate them to order level first.

---

## Merging the Tables Together

### Add product and seller details to items

Safe to merge directly. `product_id` and `seller_id` are both many:1 relative to `order_items` (verified above), so the row count doesn't change — still 112,650 rows, just with product and seller columns added.

---

In [ ]:
items_enriched = pd.merge(df_order_items, df_products, on="product_id", how="left")
items_enriched = pd.merge(items_enriched, df_sellers, on="seller_id", how="left")
print("items_enriched shape:", items_enriched.shape)

Safe to merge directly - `product_id` and `seller_id` are both many:1 relative to `order_items` (verified above), so no row-count change: still 112,650 rows, just enriched with product/seller columns.

### Aggregate items to order 

---

In [ ]:
order_items_summary = (
    items_enriched.groupby("order_id")
    .agg(
        total_items_value=("price", "sum"),
        total_freight=("freight_value", "sum"),
        num_items=("product_id", "count"),
        num_unique_products=("product_id", "nunique"),
        num_unique_sellers=("seller_id", "nunique"),
        item_mode=("product_category_name", lambda x: x.mode().iloc[0] if x.mode().size > 0 else np.nan),
        total_weight_g=("product_weight_g", "sum"),
    )
    .reset_index()
)
print("order_items_summary shape:", order_items_summary.shape)
order_items_summary.head(3)

Row count here is 98,666, not 99,441. This table only has a row for orders that actually had at least one item. That gap shows up as `NaN`s once I merge it onto the full orders table, and I dig into it in the missing-values section below.

---

### Aggregate payments to order level

---

In [ ]:
order_payments_summary = df_order_payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    payment_type_mode=("payment_type", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    max_payment_installments=("payment_installments", "max"),
).reset_index()
print("order_payments_summary shape:", order_payments_summary.shape)
order_payments_summary.head(3)

### Build the merged, order-level table

---

In [ ]:
df_merged = pd.merge(df_orders, df_customers, how="left", on="customer_id")
df_merged = pd.merge(df_merged, df_order_reviews, how="left", on="order_id")
df_merged = pd.merge(df_merged, order_items_summary, how="left", on="order_id")
df_merged = pd.merge(df_merged, order_payments_summary, how="left", on="order_id")
print("final shape:", df_merged.shape)

Row count stayed at 99,441, matching `df_orders`. That confirms none of the left joins accidentally multiplied rows — the check I run after every merge.

---

## Handling Missing Values

One look at all the nulls across the merged table, instead of patching each gap the moment it shows up.

In [ ]:
df_merged.isnull().sum()

**What the null counts say:**

- **768** orders missing `review_id` / `review_score` no label, can't train on them. Already confirmed this isn't a dedupe artifact; these orders simply never got a review.
- **775** orders missing every item-summary column - no matching row in `order_items` at all. Worth investigating: are these a clear group (e.g. cancelled orders that never shipped), and do they overlap with the 768 missing reviews? **Open - next step.**
- **2,164** missing just `item_mode` - more than the 775 with no items. So some orders *do* have items, but every item is missing its `product_category_name`, **Also open.**
- **1** order missing payment data - tiny, but worth a glance to see if it's the same kind of edge case.
- `order_approved_at` (160), `order_delivered_carrier_date` (1,783), `order_delivered_customer_date` (2,965) were already missing in the orders table itself, not caused by any merge. Probably the same not-fully-processed orders as above.
- `review_comment_title` / `review_comment_message` - mostly missing, as expected. Most reviews are just a star rating with no written comment. Free text, out of scope for a first numeric/categorical baseline.

**Next:** look at the `order_status` breakdown for the 775 (and the 2,164), and check the overlap with the 768 missing-review orders, before deciding what to drop.

---

In [ ]:
df_merged[df_merged["num_items"].isnull()]["order_status"].value_counts()

603 `unavailable` + 164 `canceled` = 767 of the 775 (99%). That confirms the guess: these orders never shipped, so there's nothing in `order_items` for them. The remaining 8 (`created`, `invoiced`, `shipped`) are a small curiosity orders that got further along but still have no item rows. Not worth chasing for a baseline.

---

Now I know these 775 are failed orders. The next question is whether they still have a `review_score` I can use, or whether they're unlabeled and have to go. I use `dropna=False` here so the no-score rows show up instead of being silently dropped from the count.

In [ ]:
df_merged[df_merged["num_items"].isnull()]["review_score"].value_counts(dropna=False)

Two very different groups were hiding inside the 775:

- **19 rows** have no items *and* no review score no label at all. These are part of the 768 unlabeled orders from earlier, not extra ones.
- **756 rows** have no items but *do* have a review score, and it's heavily skewed: **539 of 756 (71%) are 1-star.** This matches the `order_status` 

**Finding** - customers whose orders were cancelled or never arrived rated them terribly.

This matters for the drop decision: the 756 aren't just incomplete rows, they're some of the most strongly and informatively labeled data in the whole set. Dropping them (just because a merge left gaps) would throw away real signal, not just noise.


---

In [ ]:
df_merged[df_merged["total_payment_value"].isnull()]["review_score"].value_counts(dropna=False)

The one order missing payment data is also 1-star, matching the same failed-order pattern.

---

One more group to check: **items present, category missing**

The **2,164 missing** item_mode split into two groups. The **775 with no items** at all are already explained (failed orders). That leaves **~1,389 orders that do have items** but are missing every item's category.

Before I fill both with `"unknown"`, I want to confirm this second group really is ordinary orders - otherwise I'd be hiding something.


In [ ]:
mask = df_merged["item_mode"].isnull() & df_merged["num_items"].notnull()
print("orders with items but no category:", mask.sum())

In [ ]:
print(df_merged[mask]["order_status"].value_counts())

In [ ]:
print(df_merged[mask]["review_score"].value_counts(normalize=True).sort_index().round(3))

**What this shows:** of the ~1,389 orders that have items but no category, **1,332 (96%) are `delivered`** - real orders that reached the customer. Their review scores back this up: **55% are 5-star and ~75% are 4–5**, close to the overall spread and nothing like the 71% 1-star skew of the failed no-item orders.

So this is a plain metadata gap, not a failed order: the order happened normally, only the product's `product_category_name` label is missing in the products table. Filling it with `"unknown"` is safe - it won't disguise a normal order as a failure. And it doesn't need its own flag, since `has_items` already separates the orders that actually failed.

*(The 14 `canceled` / handful of `processing`/`invoiced` in this group - ~21 orders that have items but didn't complete are a minor curiosity, not chased here.)*


### Decision: how I handle the gaps

1. **Drop rows where `review_score` is null** - 768 rows (this already includes the 19 above, not extra). No label, can't train on them.
2. **For the rest,** fill the item-summary numeric columns (`total_items_value`, `total_freight`, `num_items`, `num_unique_products`, `num_unique_sellers`, `total_weight_g`) with `0`, 

fill `item_mode` with `"unknown"` (it's categorical, so `0` doesn't apply), 

`has_items` flag so the model can still tell the failed orders (no items) apart from everything else."  

3. **Same for the payment columns** (`total_payment_value`, `payment_type_mode`, `max_payment_installments`) — just 1 row, no flag needed.

*Code is present below*

---

In [ ]:
df_clean = df_merged.dropna(subset=["review_score"]).copy()
df_clean["has_items"] = df_clean["num_items"].notnull().astype(int)
item_num_cols = [
    "total_items_value",
    "total_freight",
    "num_items",
    "num_unique_products",
    "num_unique_sellers",
    "total_weight_g",
]
payment_num_cols = ["total_payment_value", "max_payment_installments"]
df_clean[item_num_cols] = df_clean[item_num_cols].fillna(0)
df_clean[payment_num_cols] = df_clean[payment_num_cols].fillna(0)
df_clean["item_mode"] = df_clean["item_mode"].fillna("unknown")
df_clean["payment_type_mode"] = df_clean["payment_type_mode"].fillna("unknown")

In [ ]:
df_clean.isna().sum()

### Dropping out-of-scope columns

`review_comment_title` / `review_comment_message` were already flagged as out of scope in the missing-values step - free text, not part of a first numeric/categorical baseline. I drop them now rather than leave mostly-empty columns sitting in `df_clean`.

---

In [ ]:
df_clean = df_clean.drop(columns=["review_comment_title", "review_comment_message"])

### Leaving the date nulls alone for now
 
Still missing: `order_approved_at` (155), `order_delivered_carrier_date` (1,746), `order_delivered_customer_date` (2,843).
 
I'm not filling these yet, for two reasons.
 
Nothing uses them yet. I haven't built a feature from these dates (raw dates cant be fed into the model we must engineer features from them and that is out of scope here), so nothing downstream breaks if they stay empty. Histograms and `.describe()` skip nulls anyway.
 
Filling them could hide a real signal. A missing delivered date probably means the order never arrived and I found earlier that those orders are mostly 1-star. If I fill in some default date, that signal disappears before I ever use it.
 
I'll revisit this when I build the delivery-days feature.
 
---

## Looking at features one at a time
 
The target (`review_score`) and each candidate feature on its own. Checklist so I can track progress instead of holding it in my head.
 
**Numeric (ready now):**
- [ ] `total_items_value`
- [ ] `total_freight`
- [ ] `num_items`
- [ ] `num_unique_products`
- [ ] `num_unique_sellers`
- [ ] `total_weight_g`
- [ ] `total_payment_value`
- [ ] `max_payment_installments`
**Categorical (ready now):**
- [ ] `item_mode`
- [ ] `payment_type_mode`
- [ ] `customer_state`
- [ ] `has_items`   


**Needs feature engineering first:** the five date columns (`order_purchase_timestamp` through `order_estimated_delivery_date`) aren't features yet they need to become something like `delivery_days` or `delivery_lateness` first. That's a bigger step, not a quick look.
 
**Out of scope for modeling:** `order_status` (only known after the fact - leakage), `review_creation_date` / `review_answer_timestamp` (post-review - leakage), `customer_city` (too granular; `customer_state` already covers geography).
 
---

In [ ]:
df_clean['review_score'].value_counts(normalize=True).sort_index().plot(kind='bar', title='Review Score Distribution', ylabel='Proportion', xlabel='Review Score')
plt.show()

**Interpretation:** 1-star reviews (11.5%) outnumber both 2-star (3.2%) and 3-star (8.2%) - the distribution isn't a smooth decline from 5 down to 1, it dips in the middle and spikes again at the bottom. This is a known pattern in review data called **extreme response bias** (sometimes visualized as a J-shaped distribution): people who had a mediocre experience rarely feel motivated to write a review at all, while people who are either delighted or angry are far more likely to. Reviewers are a self-selected group skewed toward the extremes, not a random sample of all customers.

**Why this matters for the binary threshold:** a naive "4-5 = good, 1-3 = bad" split treats 1, 2, and 3 stars as one uniform "bad" bucket. Given that 1-star reviewers may represent a categorically different (more motivated, more extreme) type of dissatisfaction than 2 or 3-star reviewers, this is worth re-examining once the threshold decision is made in the Duo phase, flagged here, not resolved yet.

### `total_items_value`

In [ ]:
sns.histplot(data=df_clean, x='total_items_value', bins=10)
plt.show()

**Problem:** almost all 98,673 rows land in the very first bin, with a long, flat, invisible tail out to the max (R$13,440). Equal-width bins split the *full range* into equal slices — when a few extreme values stretch that range enormously, the bulk of the (typical) data gets squashed into one bin. The plot is technically correct but tells us nothing about the shape of a normal order.

In [ ]:
sns.histplot(data=df_clean, x='total_items_value', bins=10, log_scale=True)
plt.show()

In [ ]:
df_clean['total_items_value'].describe()

**Interpretation:** on a log scale the shape becomes visible and looks roughly bell-shaped — a **log-normal** distribution. This is common for price-like data built from multiplicative effects (e.g. percentage markups compounding) rather than additive ones.

`describe()` backs this up numerically: mean (136.48) sits well above median (85) — consistent with the right skew seen before the log transform. The two views (shape and numbers) agree with each other, which is itself a useful sanity check.

**Open — not resolved yet:** whether the max (R$13,440) and similar high values are genuine large orders or data issues. Not deleting anything yet — that decision needs a boxplot / domain check, not just "it looks big."

### Remaining numeric features (batch scan)

Rather than repeating the full deep-dive (histogram + log scale + `describe()` + writeup) on every remaining numeric column, scanning them together first and only stopping to dig deeper on anything surprising or directly tied to the problem statement (delivery lateness -> review score).

In [ ]:
numeric_features = [
    'total_items_value',
    'total_freight',
    'num_items',
    'num_unique_products',
    'num_unique_sellers',
    'total_weight_g',
    'total_payment_value',
    'max_payment_installments',
]

df_clean[numeric_features].hist(bins=20, figsize=(15, 10))
plt.show()

**Observation:** most of these share the same heavy right-skew pattern already seen in `total_items_value` — expected, since freight, weight, and payment value are all driven by the same order-size dynamics.

**Resolved:** `num_unique_products` and `num_unique_sellers` piling up at 1 follows directly from `num_items` — most orders contain close to 1 item. This isn't just a loose correlation, it's a hard constraint: `num_unique_products` and `num_unique_sellers` can never exceed `num_items`, since you can't have more distinct products than items purchased. Worth remembering for the Group phase — these two may carry little information beyond what `num_items` already gives, a multicollinearity candidate to check properly later, not resolved here.


### Categorical features

Same tool as the target: `value_counts()` + bar chart. One thing to watch: `has_items` is binary, but `item_mode` and `customer_state` can have many distinct categories — a plain bar chart may not read well at high cardinality. Worth deciding an approach per column rather than reusing the same code blindly.

### `item_mode`

In [ ]:
df_clean["item_mode"].value_counts(normalize=True)


In [ ]:
df_clean['item_mode'].value_counts(normalize=True).cumsum().head(16)

In [ ]:
top_categories = df_clean["item_mode"].value_counts().nlargest(16).index
df_clean["item_mode"].where(df_clean["item_mode"].isin(top_categories) | df_clean["item_mode"].isin(['unknown']), other="other").value_counts().plot(kind='bar', title='Item Mode Distribution (Top 16 Categories) and Unknown value', ylabel='Counts', xlabel='Item Mode')
plt.tight_layout()
plt.show()

**Interpretation:** `other` is the single tallest bar (~17,000) — even though no individual rare category is large, the long tail combined outweighs every named category. This shows how fragmented the category space is, and is why a "top-N + other" approach is needed for modeling instead of one-hot encoding all 74 raw categories.

`unknown` (2,164 orders, ~2% of the data) is our own missing-data placeholder, not a real product category — but it isn't one uniform group. Per the earlier missing-values work, it contains two different populations: orders with no items in `order_items` at all (which skewed heavily 1-star), and orders that had real items but every one was missing `product_category_name` in the products table. Small enough overall to leave as a single placeholder for now — **flagged to revisit in the Duo phase**: if the no-items subset drives the same 1-star signal found earlier, that's worth splitting out before modeling.


### `payment_type_mode`

In [ ]:
df_clean['payment_type_mode'].value_counts(normalize=True).plot(kind='bar', title='Payment Type Mode Distribution', ylabel='Proportion', xlabel='Payment Type Mode')
plt.show()

**Interpretation:** `credit_card` dominates (76.6%), followed by `boleto` (19.9%); `debit_card` (1.5%) and `voucher` (2.0%) are minor by comparison. `not_defined` and `unknown` combined are only ~4 rows out of 98,673 — negligible, disregarded for the baseline.

Boleto (a Brazilian bank-slip payment method, typically paid as a single lump sum) versus credit card (commonly split into installments in Brazil) is a plausible explanation for the split, but this hasn't been checked against `max_payment_installments` here as an open, unverified thought rather than a confirmed finding.

### `customer_state`

In [ ]:
df_clean['customer_state'].value_counts(normalize=True)

In [ ]:
top_categories = df_clean["customer_state"].value_counts().nlargest(10).index
df_clean["customer_state"].where(
    df_clean["customer_state"].isin(top_categories),
    other="other",
).value_counts().plot(
    kind="bar",
    title="Customer State Distribution (Top 10 Categories)",
    ylabel="Counts",
    xlabel="Customer State",
)
plt.tight_layout()
plt.show()


**Interpretation:** `SP`, `RJ`, and `MG` together account for ~67% of all orders (SP alone is 42%) — unsurprising given they're Brazil's most populous and economically dominant states. The remaining 24 states form a long tail, several under 1% (e.g. `AC`, `AP`, `RR`).

**Hypothesis, not yet tested:** customers in distant/low-volume states may experience worse delivery outcomes than those in SP/RJ/MG, if sellers are concentrated near the population centers. Directly testable once `delivery_days` is engineered — group lateness/review_score by `customer_state`. **Flagged for Duo phase.**

**Backlog, not now:** `seller_state` isn't currently in `df_clean` (dropped during the item-level aggregation) — would need to be added back to actually test buyer-seller distance. A geographic map was also considered but parked — it would require reintroducing `df_geolocation` (deliberately excluded in the Shape phase) for a visual that a bar chart already covers functionally.


### `has_items`

In [ ]:
df_clean['has_items'].value_counts(normalize=True).plot(kind='bar', title='Has Items Distribution', ylabel='Proportion', xlabel='Has Items')
plt.show()

**Interpretation:** `has_items` is heavily imbalanced — 756 of 98,673 orders (~0.77%) have no matching row in `order_items` at all, matching the count found back in the missing-values work. This isn't new information on its own, but it's a useful cross-check that the flag column built during cleanup behaves as expected.

Already established earlier that this small group skews heavily toward 1-star reviews (71%) — worth carrying `has_items` forward as a candidate feature into Duo, even though the split is extreme, because the signal inside it is strong.


## Exploring Feature Relationships with the Target

Each candidate feature vs. review_score. Use this to decide the binary threshold (which stars = good vs. bad).

## Checking for Redundant Features

Check redundancy among the features that survived the previous step before they go into feature engineering.

## Findings Summary

- Binary threshold decided:
- Features to carry into 02_preprocessing:
- Data quality issues to handle: